<a href="https://colab.research.google.com/github/virwang/Fanshawe_DataVisualization26W/blob/main/Capstone_Group04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO-6151-(01)-26W Data Visualization for Machine Learning
## Capstone Project: Impact of Geopolitical Conflicts and Trade Policies on US Market Sectors
**Date:** March 28th, 2026

**Group Members:**
* Yun-Jiung Wang (1256222)
* Andrew Polk (1161715)
* Ebenebe, Kaosicho (1252551)

---

### 1. Project Objective
This project analyzes and visualizes the resilience of different US industrial sectors under the dual pressure of Global Tariff Policies and the 2026 Iranian Conflict. We utilize the `yfinance` library to extract financial data and evaluate how specific Exchange Traded Funds (ETFs) and market sentiment indicators respond to these macroeconomic and geopolitical shocks.

### 2. Research Scope
We focus on five key assets and one sentiment benchmark to represent diverse segments of the global economy:

* **SPY:** S&P 500 (Broad Market Benchmark)
* **QQQ:** Nasdaq 100 (Technology & Innovation)
* **ITA:** Aerospace & Defense (National Security)
* **ICLN:** Clean Energy (Future Sustainability)
* **VDE:** Vanguard Energy (Traditional Oil & Gas)
* **VIX:** CBOE Volatility Index (Market Sentiment & "Fear Gauge")

### 3. Methodology & Technical Implementation
* **Data Acquisition:** Extracted historical adjusted closing prices using Python and `yfinance`.
* **Event Study Analysis:** Mapped specific dates of tariff announcements (Policy Risk) and military escalations (Conflict Risk) against market performance.
* **Time-Series Visualization:** Developed interactive dashboards using Plotly Dash to track cumulative returns and rolling volatility.
* **Predictive Modeling:** Implemented an XGBoost Regression model to predict market volatility (VIX) trends using lagged sector returns.

### 4. Project Motivation
* **Geopolitical Risk Analysis:** We classify "Global Tariffs" as strategic policy risk and the "Iranian Conflict" as direct conflict risk. This study examines how markets react differently to these two types of instability.
* **Sector Divergence:** During crises, Defense (ITA) and Energy (VDE) often act as safe-haven assets, while Technology (QQQ) may underperform due to trade barrier costs.
* **Predictive Insights:** We apply Machine Learning to identify which industrial sectors serve as the most reliable leading indicators for broader market panic.

### 5. Expected Deliverables
* **Interactive Dashboard:** A Plotly Dash interface highlighting performance changes and VIX spikes during key events.
* **Risk Assessment Metrics:** Calculation of maximum drawdowns and correlation matrices to measure hedging effectiveness.
* **XGBoost Feature Importance:** Visual analysis identifying which sector contributes most to the accuracy of the volatility prediction model.

In [1]:
!pip install dash
!pip install yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 36.3 MB/s eta 0:00:00


## **0. Import Libs**

In [18]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import requests

from datetime import datetime
from dateutil.relativedelta import relativedelta
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output
from google.colab import output

import xgboost as xgb
from sklearn.metrics import mean_squared_error
import plotly.express as px

## **Phase 1: Data Preparation and Feature Engineering**


Instead of using raw price levels, we utilize Daily Percentage Returns as our primary features. This ensures data stationarity, which is a requirement for most machine learning algorithms. By calculating the relative change against the previous day's closing price, the model focuses on market volatility and momentum rather than long-term price trends. The inclusion of lagged features (T-1 to T-5) allows the XGBoost model to detect short-term behavioral patterns triggered by geopolitical events.

### **1. Data Collection:**

1.1. Define required tickers

1.2. Download the raw data

1.3. Display first 2 rows

In [30]:
# Define the Stocks for observation
STOCKS = {
    "SPY": "Market",
    "QQQ": "Tech",
    "ITA": "Defense",
    "ICLN": "Clean Energy",
    "VDE": "Energy"
}

# Define date region
today = datetime.today()
START= (today - relativedelta(years=2)).strftime("%Y-%m-%d")
END = datetime.today().strftime("%Y-%m-%d")

tickers = list(STOCKS.keys()) + ["^VIX"]
print(f"\nLoad data from {START} to {END}")
# Download the dataset
df_raw = yf.download(tickers, start=START, end=END, auto_adjust=True)

df_raw.head(2)

[*********************100%***********************]  6 of 6 completed


Load data from 2024-03-29 to 2026-03-29


Price           Close                                                         \
Ticker           ICLN         ITA         QQQ         SPY         VDE   ^VIX   
Date                                                                           
2024-04-01  13.495602  129.119003  440.671417  509.754089  125.401146  13.65   
2024-04-02  13.273411  128.467926  436.868317  506.512970  127.065971  14.61   

Price            High                                      ...        Open  \
Ticker           ICLN         ITA         QQQ         SPY  ...         QQQ   
Date                                                       ...               
2024-04-01  13.621187  130.411276  443.226595  511.921376  ...  440.691214   
2024-04-02  13.437638  128.931568  437.185247  506.649598  ...  435.848222   

Price                                       Volume                    \
Ticker             SPY         VDE   ^VIX     ICLN     ITA       QQQ   
Date                                                                   
2024-04-01  511.384455  124.918715  13.61  3284300  238200  38729000   
2024-04-02  505.927190  126.006541  13.74  5033400  685800  44259700   

Price                              
Ticker           SPY     VDE ^VIX  
Date                               
2024-04-01  62477500  658000    0  
2024-04-02  74230300  528700    0  

[2 rows x 30 columns]

### **2. Data Preprocessing:**


#### **2.1. Flattening MultiIndex Data:**
<p>

  The initial dataset retrieved from yfinance contains a MultiIndex structure, which is complex for direct visualization. To simplify our workflow, we extracted only the adjusted closing prices and flattened the column headers. This process ensures the DataFrame is compatible with Plotly and easier for team members to perform further calculations, such as daily returns or volatility analysis.
</p>


In [11]:
# Get the Adj close data
df_final = df_raw['Close'].copy()

# Remove label name 'Ticker'
df_final.columns.name = None

# Convert Date to normal columns
df_final = df_final.reset_index()

# Rename ^VIX to VIX
df_final = df_final.rename(columns={'^VIX': 'VIX'})

# Verification:
print("Transformed DataFrame (Single Level Column):")
print(df_final.head())
print("\nColumns list:", df_final.columns.tolist())

Transformed DataFrame (Single Level Column):
        Date       ICLN         ITA         QQQ         SPY         VDE  \
0 2024-04-01  13.495602  129.119003  440.671417  509.754089  125.401146   
1 2024-04-02  13.273411  128.467911  436.868347  506.513031  127.065956   
2 2024-04-03  13.360355  127.905647  437.848785  507.069397  128.078110   
3 2024-04-04  13.350695  128.408752  431.153748  500.880096  127.917305   
4 2024-04-05  13.205790  129.572769  436.234497  506.112732  129.307816   

         VIX  
0  13.650000  
1  14.610000  
2  14.330000  
3  16.350000  
4  16.030001  

Columns list: ['Date', 'ICLN', 'ITA', 'QQQ', 'SPY', 'VDE', 'VIX']


#### **2.2 Data Normalization (Base 100):**

For visualization, we need to normalize the data to make them comparable.

In [12]:
df_norm = df_final.copy()

# remove the date column for normalization
cols_to_norm = [col for col in df_norm.columns if col != 'Date']

for col in cols_to_norm:
    # make the first day's value 100
    df_norm[col] = (df_norm[col] / df_norm[col].iloc[0]) * 100

print("Normalization Complete. All assets now start at 100.")

Normalization Complete. All assets now start at 100.


### **2.3. Calculate Returns**

Convert prices to percentage change for machine learning compatibility

In [17]:
# Set Date as index for time series operations
df_ml = df_final.set_index('Date').copy()
df_returns = df_ml.pct_change().dropna()

display(df_returns.head(2))
display(df_ml.head(2))

,ICLN,ITA,QQQ,SPY,VDE,VIX
Date,,,,,,
2024-04-02,-0.016464,-0.005043,-0.008630,-0.006358,0.013276,0.070330
2024-04-03,0.006550,-0.004377,0.002244,0.001098,0.007966,-0.019165


,ICLN,ITA,QQQ,SPY,VDE,VIX
Date,,,,,,
2024-04-01,13.495602,129.119003,440.671417,509.754089,125.401146,13.65
2024-04-02,13.273411,128.467911,436.868347,506.513031,127.065956,14.61


#### **2.4. Create Lagged Features:**

Generate features representing the past 5 days of market activity


In [26]:
lags = 5
df_features = df_returns.copy()

for col in df_returns.columns:
    for i in range(1, lags + 1):
        df_features[f'{col}_lag_{i}'] = df_returns[col].shift(i)

# Remove rows with missing values caused by the shifting operation
df_ml_ready = df_features.dropna()

#### **2.5 Define Features (X) and Target (y)**

In [25]:
# Target variable: VIX
y = df_ml.loc[df_ml_ready.index, 'VIX']

# Feature variables: All sector returns and their lagged values (excluding current day VIX)
X = df_ml_ready.drop(columns=['VIX'])

print(f"Feature Engineering Complete. Matrix shape: {X.shape}")

Feature Engineering Complete. Matrix shape: (494, 35)


## **Phase 2: XGBoost Model Training and Feature Importance Analysis**
We split the data chronologically to train the XGBoost Regressor. Following the training, we visualize the feature importance to determine which sectors are the strongest predictors of market volatility.

### **1. Split train/test data**

In [33]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

### **2. Initialize and Train XGBoost Regressor**

In [34]:
model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    objective='reg:squarederror',
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

### **3. Evaluate Performance**

In [35]:
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"Model Training Complete. Root Mean Squared Error (RMSE): {rmse:.4f}")

Model Training Complete. Root Mean Squared Error (RMSE): 3.6047


Price Data for Event Analysis

In [ ]:
# Separate ETF prices and VIX series for modules
prices = df_close[list(STOCKS.keys())].copy()
prices.index = pd.to_datetime(prices.index)
prices.dropna(how="all", inplace=True)

# VIX into its own series
vix = df_close["VIX"].copy()
vix.index = pd.to_datetime(vix.index)

print(f"Prices shape: {prices.shape}")
print(f"Range: {prices.index[0].date()} -> {prices.index[-1].date()}")
prices.tail(5)

Prices shape: (561, 5)
Range: 2024-01-02 -> 2026-03-27


,SPY,QQQ,ITA,ICLN,VDE
Date,,,,,
2026-03-23,655.380005,588.000000,223.350006,18.180000,169.429993
2026-03-24,653.179993,583.979980,222.639999,18.309999,172.119995
2026-03-25,656.820007,587.820007,225.860001,18.809999,171.440002
2026-03-26,645.090027,573.789978,220.119995,18.030001,174.139999
2026-03-27,634.090027,562.580017,216.039993,17.850000,176.949997


## Event Definitions

In [ ]:
EVENTS = {
    # Iran War Dates:
    "Israel strikes Iran": "2026-02-28",
    # Trump Tariff Dates:
    "Tariff":"2025-04-02"
}

## Iran War Module

In [ ]:
# Pull Iran date from EVENTS
IRAN_DATE = pd.Timestamp(next(v for k, v in EVENTS.items() if "Iran" in k))

# Get event context
WINDOW_START = IRAN_DATE - pd.Timedelta(days=10)
WINDOW_END   = min(IRAN_DATE + pd.Timedelta(days=30), prices.index[-1])

iran_prices = prices.loc[WINDOW_START:WINDOW_END].copy()

# Normalizes all ETF prices to 0% on the day before the strike so all lines start at the same baseline
pre_strike = prices.loc[:IRAN_DATE].iloc[-2]
iran_norm  = ((iran_prices / pre_strike) -1) * 100

# VIX series for the same window
iran_vix = vix.loc[WINDOW_START:WINDOW_END]

# Defines sector groupings
SECTORS = {
    "All":          ["SPY", "QQQ", "ITA", "ICLN", "VDE"],
    "Broad Market": ["SPY", "QQQ"],
    "Defense":      ["ITA"],
    "Energy":       ["ICLN", "VDE"],
}

# Defines ETF colours
COLORS = {
    "SPY":  "#1D9E75",
    "QQQ":  "#3C3489",
    "ITA":  "#534AB7",
    "ICLN": "#2CA02C",
    "VDE":  "#EF8B2C",
}

# Function to build the stacked subplot chart
def build_iran_fig(selected_sector):
    tickers_sel = SECTORS[selected_sector]
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.06,
        subplot_titles=["Sector % Change from Pre-Event Close", "VIX (Market Fear Index)"],
    )

    # First row: sector ETF with normalized returns
    for ticker in tickers_sel:
        if ticker not in iran_norm.columns:
            continue
        fig.add_trace(go.Scatter(
            x=iran_norm.index, y=iran_norm[ticker],
            name=f"{ticker} - {STOCKS[ticker]}",
            line=dict(color=COLORS[ticker], width=2),
            hovertemplate=f"<b>{ticker}</b><br>%{{x|%b %d %Y}}<br>%{{y:.1f}}%<extra></extra>",
        ), row=1, col=1)

    # Second row: VIX plot
    fig.add_trace(go.Scatter(
        x=iran_vix.index, y=iran_vix.values,
        name="VIX", showlegend=True,
        line=dict(color="gray", width=1.5),
        fill="tozeroy", fillcolor="rgba(180,180,180,0.15)",
        hovertemplate="<b>VIX</b><br>%{x|%b %d %Y}<br>%{y:.1f}<extra></extra>",
    ), row=2, col=1)

    # Event line + shading on both rows
    for row in [1, 2]:
        fig.add_vline(x=IRAN_DATE.timestamp()*1000,
                      line=dict(color="red", width=2, dash="dash"), row=row, col=1)
    fig.add_vrect(x0=IRAN_DATE, x1=IRAN_DATE+pd.Timedelta(days=7),
                  fillcolor="red", opacity=0.07, line_width=0)

    # Baseline on row 1
    fig.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"), row=1, col=1)

    # Event label
    visible = [t for t in tickers_sel if t in iran_norm.columns]
    fig.add_annotation(
        x=IRAN_DATE, y=iran_norm[visible].max().max(),
        text="  Israel strikes Iran",
        showarrow=False, font=dict(size=11, color="red"),
        xanchor="left", row=1, col=1,
    )

    fig.update_yaxes(title_text="% change", row=1, col=1)
    fig.update_yaxes(title_text="VIX", row=2, col=1)
    fig.update_xaxes(tickformat="%b %d", tickangle=45, row=2, col=1)
    fig.update_layout(
        title=f"Iran War - Sector: {selected_sector}",
        hovermode="x unified",
        template="plotly_white",
        margin=dict(r=40),
        height=550,
    )
    return fig

app_iran = Dash(__name__)

app_iran.layout = html.Div([
    html.H2("Iran-Israel Conflict: Market Reaction by Sector"),
    html.P(f"% change from close on {(IRAN_DATE - pd.Timedelta(days=1)).strftime('%b %d %Y')}"),
    html.Label("View by sector:"),
    dcc.Dropdown(
        id="sector-dropdown",
        options=[{"label": k, "value": k} for k in SECTORS.keys()],
        value="All", clearable=False, style={"width": "300px"},
    ),
    dcc.Graph(id="iran-chart"),
])

@app_iran.callback(Output("iran-chart", "figure"), Input("sector-dropdown", "value"))
def update_iran_chart(selected_sector):
    return build_iran_fig(selected_sector)

#app_iran.run(port=8050, debug=False)
#output.serve_kernel_port_as_iframe(8050)

## Trump Tariffs Module

In [ ]:
TARIFF_DATE = pd.Timestamp(next(v for k, v in EVENTS.items() if "Tariff" in k))

TARIFF_WINDOW_START = TARIFF_DATE - pd.Timedelta(days=30)
TARIFF_WINDOW_END   = min(TARIFF_DATE + pd.Timedelta(days=60), prices.index[-1])

tariff_prices = prices.loc[TARIFF_WINDOW_START:TARIFF_WINDOW_END].copy()

pre_tariff  = prices.loc[:TARIFF_DATE].iloc[-1]
tariff_norm = ((tariff_prices / pre_tariff) - 1) * 100

tariff_vix = vix.loc[TARIFF_WINDOW_START:TARIFF_WINDOW_END]

TARIFF_EVENTS = {
    "Liberation Day":      pd.Timestamp("2025-04-02"),
    "90-day Pause":        pd.Timestamp("2025-04-09"),
    "US-China Escalation": pd.Timestamp("2025-04-15"),
}

TARIFF_SECTORS = {
    "All":          ["SPY", "QQQ", "ITA", "ICLN", "VDE"],
    "Broad Market": ["SPY", "QQQ"],
    "Defense":      ["ITA"],
    "Energy":       ["ICLN", "VDE"],
}

def build_tariff_fig(selected_sector):
    tickers_sel = TARIFF_SECTORS[selected_sector]
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.06,
        subplot_titles=["Sector % Change from Pre-Announcement Close", "VIX (Market Fear Index)"],
    )

    # First row: sector ETF with normalized returns
    for ticker in tickers_sel:
        if ticker not in tariff_norm.columns:
            continue
        fig.add_trace(go.Scatter(
            x=tariff_norm.index, y=tariff_norm[ticker],
            name=f"{ticker} - {STOCKS[ticker]}",
            line=dict(color=COLORS[ticker], width=2),
            hovertemplate=f"<b>{ticker}</b><br>%{{x|%b %d %Y}}<br>%{{y:.1f}}%<extra></extra>",
        ), row=1, col=1)

    # Second row: VIX plot
    fig.add_trace(go.Scatter(
        x=tariff_vix.index, y=tariff_vix.values,
        name="VIX", showlegend=True,
        line=dict(color="gray", width=1.5),
        fill="tozeroy", fillcolor="rgba(180,180,180,0.15)",
        hovertemplate="<b>VIX</b><br>%{x|%b %d %Y}<br>%{y:.1f}<extra></extra>",
    ), row=2, col=1)

    # Event lines on both rows
    event_colors = {"Liberation Day": "red", "90-day Pause": "green", "US-China Escalation": "orange"}
    visible = [t for t in tickers_sel if t in tariff_norm.columns]
    y_range = tariff_norm[visible].max().max() - tariff_norm[visible].min().min() if visible else 10
    y_min   = tariff_norm[visible].min().min() if visible else -5
    y_offsets = [0.85, 0.70, 0.55]

    for (label, date), y_frac in zip(TARIFF_EVENTS.items(), y_offsets):
        color = event_colors.get(label, "gray")
        y_pos = y_min + y_frac * y_range
        for row in [1, 2]:
            fig.add_vline(x=date.timestamp()*1000,
                          line=dict(color=color, width=1.5, dash="dash"), row=row, col=1)
        fig.add_annotation(x=date, y=y_pos, text=f" {label}",
                           showarrow=False, font=dict(size=10, color=color),
                           xanchor="left", bgcolor="white", borderpad=2,
                           row=1, col=1)

    fig.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"), row=1, col=1)
    fig.update_yaxes(title_text="% change", row=1, col=1)
    fig.update_yaxes(title_text="VIX", row=2, col=1)
    fig.update_xaxes(tickformat="%b %d", tickangle=45, row=2, col=1)
    fig.update_layout(
        title=f"Trump Tariffs - Sector: {selected_sector}",
        hovermode="x unified",
        template="plotly_white",
        margin=dict(r=40),
        height=550,
    )
    return fig

app_tariff = Dash(__name__)

app_tariff.layout = html.Div([
    html.H2("Trump Tariffs (2025): Market Reaction by Sector"),
    html.P(f"% change from close on {(TARIFF_DATE - pd.Timedelta(days=1)).strftime('%b %d %Y')}"),
    html.Label("View by sector:"),
    dcc.Dropdown(
        id="tariff-sector-dropdown",
        options=[{"label": k, "value": k} for k in TARIFF_SECTORS.keys()],
        value="All", clearable=False, style={"width": "300px"},
    ),
    dcc.Graph(id="tariff-chart"),
])

@app_tariff.callback(Output("tariff-chart", "figure"), Input("tariff-sector-dropdown", "value"))
def update_tariff_chart(selected_sector):
    return build_tariff_fig(selected_sector)

#app_tariff.run(debug=True)

## Preprocessing

In [ ]:
# Feature Engineering
import plotly.express as px
from sklearn.preprocessing import StandardScaler

# Log daily returns (no VIX)
log_returns = np.log(prices / prices.shift(1)).dropna()

# 20-day rolling volatility (annualized)
rolling_vol = log_returns.rolling(20).std() * np.sqrt(252)

# RSI (14-day)
def compute_rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

rsi = prices.apply(compute_rsi)

# MACD (12/26 EMA diff)
def compute_macd(series):
    ema12 = series.ewm(span=12, adjust=False).mean()
    ema26 = series.ewm(span=26, adjust=False).mean()
    return ema12 - ema26

macd = prices.apply(compute_macd)

print("Features engineered: log_returns, rolling_vol, RSI, MACD")
print(f"  log_returns shape: {log_returns.shape}")

# Cross-Sector Correlation
# Include VIX log returns for correlation context
log_returns_vix = pd.concat([log_returns, np.log(vix / vix.shift(1)).rename("VIX")], axis=1).dropna()

snapshots_tariff = {
    "Pre-Tariff (Jan–Mar 2025)":  ("2025-01-01", "2025-04-01"),
    "Post-Tariff (Apr–Jun 2025)": ("2025-04-02", "2025-07-01"),
}
snapshots_iran = {
    "Pre-Iran (Dec 2025–Feb 2026)": ("2025-12-01", "2026-02-27"),
    "Post-Iran (Mar 2026–now)":     ("2026-02-28", prices.index[-1].strftime("%Y-%m-%d")),
}

corr_data_tariff = {}
for label, (start, end) in snapshots_tariff.items():
    w = log_returns_vix.loc[start:end]
    if len(w) > 5:
        corr_data_tariff[label] = w.corr().round(2)

corr_data_iran = {}
for label, (start, end) in snapshots_iran.items():
    w = log_returns_vix.loc[start:end]
    if len(w) > 5:
        corr_data_iran[label] = w.corr().round(2)

print("Correlation snapshots:", list(corr_data_tariff.keys()) + list(corr_data_iran.keys()))

# Before and after scaling
SCALE_TICKER = "SPY"
raw_series = log_returns[SCALE_TICKER].dropna()
scaler_std = StandardScaler()
scaled_std = scaler_std.fit_transform(raw_series.values.reshape(-1, 1)).flatten()

fig_scale = make_subplots(rows=1, cols=2, subplot_titles=["Raw Log Returns", "StandardScaler"])
for col_idx, (data, color) in enumerate([
    (raw_series.values, "steelblue"),
    (scaled_std, "darkorange"),
], start=1):
    fig_scale.add_trace(
        go.Histogram(x=data, nbinsx=60, marker_color=color, showlegend=False),
        row=1, col=col_idx,
    )
fig_scale.update_layout(
    title_text=f"{SCALE_TICKER} Log Return Distribution - Raw vs Standardized",
    template="plotly_white",
)
fig_scale.show()

Features engineered: log_returns, rolling_vol, RSI, MACD
  log_returns shape: (560, 5)
Correlation snapshots: ['Pre-Tariff (Jan–Mar 2025)', 'Post-Tariff (Apr–Jun 2025)', 'Pre-Iran (Dec 2025–Feb 2026)', 'Post-Iran (Mar 2026–now)']


## Model

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats

FEATURE_TICKERS = list(STOCKS.keys())

frames = []
for tk in FEATURE_TICKERS:
    df_tk = pd.DataFrame({
        "log_ret": log_returns[tk], # todays log return
        "rolling_vol": rolling_vol[tk], # how volatile the last 20 days were
        "rsi": rsi[tk], # momentum indicator 0-100
        "macd": macd[tk], # # short vs long term trend difference
        "target": log_returns[tk].shift(-5), # what we are trying to predict, return _ days from now
    }).dropna()
    df_tk["ticker"] = tk
    frames.append(df_tk)

# stack all tickers into one dataframe
model_df = pd.concat(frames).dropna()

# sort by date
model_df = model_df.sort_index()

# train on 2024, test on 2025 onwards (test predictive ability)
# both events are in the test
train_df = model_df.loc[:"2024-12-31"]
test_df  = model_df.loc["2025-01-01":]

FEATURES = ["log_ret", "rolling_vol", "rsi", "macd"]

# separate features from target
X_train = train_df[FEATURES].values
y_train = train_df["target"].values
X_test  = test_df[FEATURES].values
y_test  = test_df["target"].values

# scale features
scaler_model = StandardScaler()
X_train = scaler_model.fit_transform(X_train)
X_test  = scaler_model.transform(X_test)

# train model
# XGBoost can learn curved and conditional relationships that occur in financial data
model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

# test
y_pred = model.predict(X_test)

r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f"XGBoost  |  R2: {r2:.4f}  |  MSE: {mse:.6f}")

residuals = y_test - y_pred

# predicted vs actual scatter and residuals side by side
fig_resid = make_subplots(rows=1, cols=2, subplot_titles=["Predicted vs Actual", "Residuals"])
fig_resid.add_trace(go.Scatter(x=y_test, y=y_pred, mode="markers",
    marker=dict(color="steelblue", opacity=0.5, size=4), name="Pred vs Actual"), row=1, col=1)
fig_resid.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()],
    mode="lines", line=dict(color="red", dash="dash"), name="Perfect fit"), row=1, col=1)
fig_resid.add_trace(go.Scatter(x=y_pred, y=residuals, mode="markers",
    marker=dict(color="darkorange", opacity=0.5, size=4), name="Residuals"), row=1, col=2)
fig_resid.add_hline(y=0, line=dict(color="red", dash="dash"), row=1, col=2)

fig_resid.update_xaxes(title_text="Actual Return", row=1, col=1)
fig_resid.update_yaxes(title_text="Predicted Return", row=1, col=1)
fig_resid.update_xaxes(title_text="Predicted Return", row=1, col=2)
fig_resid.update_yaxes(title_text="Residual (Actual - Predicted)", row=1, col=2)
fig_resid.update_layout(title="Model Diagnostics", template="plotly_white", showlegend=False)
fig_resid.show()

# checks if residuals follow a normal distribution
qq = stats.probplot(residuals, dist="norm")
qq_x = list(qq[0][0])
qq_y = list(qq[0][1])

fig_qq = go.Figure()
fig_qq.add_trace(go.Scatter(x=qq_x, y=qq_y, mode="markers",
    marker=dict(color="steelblue", size=4), name="Residuals"))
fig_qq.add_trace(go.Scatter(x=[min(qq_x), max(qq_x)],
    y=[qq[1][1]+qq[1][0]*min(qq_x), qq[1][1]+qq[1][0]*max(qq_x)],
    mode="lines", line=dict(color="red"), name="Normal line"))
fig_qq.update_layout(title="Q-Q Plot of Residuals",
    xaxis_title="Theoretical Quantiles", yaxis_title="Sample Quantiles",
    template="plotly_white")
fig_qq.show()

XGBoost  |  R2: -0.1648  |  MSE: 0.000235


## SHAP

In [ ]:
import shap

# create the explainer
explainer = shap.TreeExplainer(model)

# compute shap values for every test sample
# each value represents how much a feature pushed the prediction up or down
shap_values = explainer.shap_values(X_test)

# average the absolute shap values across all test samples
# gives us a single importance score per feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)

fig_shap_bar = go.Figure(go.Bar(
    x=mean_abs_shap, y=FEATURES, orientation="h", marker_color="steelblue",
))
fig_shap_bar.update_layout(
    title="SHAP Feature Importance (mean |SHAP value|)",
    xaxis_title="Mean |SHAP|",
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
)
fig_shap_bar.show()

# test sample
SAMPLE_IDX = 0

# waterfall chart for a single prediction
fig_shap_sample = go.Figure(go.Bar(
    x=shap_values[SAMPLE_IDX], y=FEATURES, orientation="h",
    marker_color=["crimson" if v < 0 else "steelblue" for v in shap_values[SAMPLE_IDX]],
))
fig_shap_sample.update_layout(
    title=f"SHAP Waterfall — Test Sample {SAMPLE_IDX}",
    xaxis_title="SHAP value (impact on predicted return)",
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
)
fig_shap_sample.show()

# the average prediction the model makes before any features are considered
print(f"Base value (expected output): {explainer.expected_value:.6f}")

## Dash App